# Module 01 — Spatial structural change

**Where did European automotive employment go, 2008–2023?**

### The framing

**Pavlínek (2019)** explains the changing geography of the European auto industry through Harvey's *uneven development* and the *spatio-temporal fix*: firms pursue spatial profit-seeking, so the industry grows in **integrated peripheries** while **restructuring in the traditional core**. His evidence is 2,124 restructuring *events* (2005–2016) plus interviews.

**Lefèvre et al. (2022)** argue that transitions should be read through *structural change* — the evolving **composition** of output and employment — not through aggregates.

**Our contribution:** Pavlínek tests his claim on *events*. We test it on **stocks** — actual regional employment levels — with an explicit decomposition. Shift-share separates *“this region shrank because the industry shrank”* from *“this region lost ground relative to the industry”*, which is exactly the core-vs-periphery question.

### Three commitments

1. **Balanced sets only.** A region missing an endpoint is dropped, never zero-filled. `balance_report()` shows what each window costs.
2. **France = URSSAF (Rule 1).** URSSAF's 13 régions are French **NUTS-1** (`FR1`, `FRB`…`FRM`), *not* NUTS-2 (still the 22 pre-2016 régions). So France can only be spliced at **NUTS-1**. Measurement bases differ (*effectifs salariés* vs *persons employed*) — flagged wherever they meet.
3. **Suppression is not random.** Eurostat withholds cells exactly in regions dominated by one or two big plants. Coverage is reported with every result.

In [ ]:
import os
# Reload edited .py modules automatically — without this, a kernel that
# imported an older version keeps serving it (AttributeError on new functions).
%load_ext autoreload
%autoreload 2

import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import foundation as fd
import module01_spatial as m1
# --- archive bootstrap: resolve paths from the project root ---------------
# The original was CONFIG.data_dir = './data', which assumed the notebook ran
# beside a data/ folder. In the archived layout the inputs live in
# 02_data_inputs/, so the path is taken from specular_paths instead and this
# notebook works from wherever it is opened.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent))   # 01_code/
from specular_paths import P
fd.CONFIG.data_dir = str(P.inputs)
fd.CONFIG.out_dir  = str(P.outputs)
print('data:', fd.CONFIG.data_dir)
fd.CONFIG.out_dir  = './outputs'
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

## 1. Build the panels

`level=1` (NUTS-1) is the level where France can be included. We also build C27 (electrical equipment) for the two-sector decomposition, and C29 *local units* for average establishment size.

In [ ]:
p29 = m1.european_panel('C29', 'employment', level=1, france='urssaf', drop_other_bloc=True)
p27 = m1.european_panel('C27', 'employment', level=1, france='urssaf', drop_other_bloc=True)
pu  = m1.european_panel('C29', 'units',      level=1, france='exclude', drop_other_bloc=True)

print('regions:', p29['geo_code'].nunique(), '| sources:', p29['source'].value_counts().to_dict())
print('regions per bloc:', p29.groupby('bloc')['geo_code'].nunique().to_dict())

## 2. Choose the window — *before* computing anything

A long window with 60% coverage tells you less than a short one with 95%. Pick the window from this table, not from habit.

In [ ]:
br = m1.balance_report(p29)
display(br.round(1))
print('Chosen: 2012–2023 — 76 regions, ~92% of 2012 and ~97% of 2023 totals.')

## 3. Headline — the bloc reallocation

Panel (b), the **shares**, is the structural-change statement in Lefèvre's sense: the composition of European automotive employment, not its total.

In [ ]:
traj = m1.bloc_trajectories(p29, balanced=True, y0=2012, y1=2023)
fig = m1.fig_bloc_trajectories(traj, save=str(fd.CONFIG.out('fig01_bloc_trajectories.png')), title_note='2012–2023')
plt.show()

display(traj.pivot(index='year', columns='bloc', values='share_pct').round(1))

## 4. Did each region keep pace with the European industry?

Shift-share splits a region's change into two parts:

$$\underbrace{\Delta E_i}_{\text{what happened}} = \underbrace{E_{i0}\,g}_{\text{if it grew at the EU rate}} + \underbrace{E_{i0}(g_i-g)}_{\textbf{the gap}}$$

**The gap is the result.** A region can *add* jobs and still lose ground, if it added them more slowly than the industry as a whole. Panel (a) shows the arithmetic on four regions; panel (b) shows the gap for all of them, in jobs, with readable names.

In [ ]:
ss = m1.shift_share_1sector(p29, 2012, 2023)
print(f"EU-wide growth {100*ss.attrs['eu_growth']:+.1f}% over {ss.attrs['n_regions']} balanced regions\n")
fig = m1.fig_shift_share_explained(ss, panel=p29, save=str(fd.CONFIG.out('fig01_shift_share.png'))); plt.show()

display((ss.groupby('bloc')[['eu_growth_component','differential_shift','actual_change']].sum()/1000).round(1))

In [ ]:
# region-level detail
cols = ['geo_code','country_iso2','bloc','E0','E1','growth_rate','differential_shift']
print('TOP 10 GAINERS'); display(ss.head(10)[cols].round(2))
print('TOP 10 LOSERS');  display(ss.tail(10)[cols].round(2))
ss.to_csv('outputs/tbl01_shift_share_2012_2023.csv', index=False)

## 5. Was it the sector, or the region?

Section 4 showed *that* regions diverged. This asks *why*, using a second sector (NACE C27, electrical equipment) as the comparison.

A bloc's change in vehicle jobs splits into three parts that add up exactly:

| part | the question it answers |
|---|---|
| **national growth** | what if it had simply grown with the whole economy? |
| **sector effect** (industry mix) | was it hurt just by *being in vehicles* rather than electrical equipment? |
| **regional effect** (competitive) | did it do better or worse *at vehicles* than everyone else? |

Panel (a) walks the worst-hit bloc through the arithmetic as a waterfall. Panel (b) puts the two parts that carry meaning side by side.

> Read the answer off panel (b): the sector effect is small and negative everywhere, the regional effect is large and points in **opposite directions**. Being in vehicles was a mild drag on everyone; *where* production happened is what actually separated the blocs.

In [ ]:
ss2 = m1.shift_share_2sector(p29, p27, 2012, 2023)
print('growth  all sectors: %+.1f%%' % (100*ss2.attrs['G_total']),
      '| by sector:', {k: f'{100*v:+.1f}%' for k, v in ss2.attrs['G_by_sector'].items()})

display((ss2.groupby(['bloc','nace'])[['national_growth','industry_mix','competitive','actual_change']].sum()/1000).round(1))
fig = m1.fig_shift_share_2sector(ss2, panel=p29, save=str(fd.CONFIG.out('fig01_shift_share_2sector.png'))); plt.show()

## 6. Is the industry spreading out, or bunching up?

Pavlínek's expectation is **dispersal** toward the peripheries. Three ways to check, in two figures.

**Figure 1 — the Herfindahl index.** Panel (a) is the whole industry, with arrows saying which direction means what. Panel (b) computes the same index *within* each bloc, which asks the sharper question: is the core thinning out while the east builds hubs of its own?

**Figure 2 — the same question without an index:** the share held by the largest regions against thresholds anyone can read, and how many regions it takes to make up half the industry.

> The residual `other` bloc is left out of the bloc panel. It is a bookkeeping category and at NUTS-1 often a single region, which pins its index at the 10,000 ceiling and flattens every real line.

In [ ]:
conc      = m1.concentration(p29, 2012, 2023)
conc_bloc = m1.concentration_by_bloc(p29, 2012, 2023)

fig = m1.fig_hhi(conc, conc_bloc, panel=p29,
                 save=str(fd.CONFIG.out('fig01_hhi.png'))); plt.show()

fig = m1.fig_concentration_detail(conc, panel=p29,
                                  save=str(fd.CONFIG.out('fig01_concentration.png'))); plt.show()

display(conc.round(1))
print('\nHHI within each bloc:')
display(conc_bloc.pivot(index='year', columns='bloc', values='hhi').round(0))

In [ ]:
fig = m1.fig_lorenz(p29, years=(2012, 2023),
                   save=str(fd.CONFIG.out('fig01_lorenz.png'))); plt.show()

for y in (2012, 2023):
    lz = m1.lorenz_points(p29, y)
    print(f'{y}: Gini {lz.attrs["gini"]:.3f} over {lz.attrs["n_regions"]} regions')

## 7. Average establishment size

Employment ÷ local units. Rising = consolidation into large plants; falling = fragmentation toward a supplier-dense tissue. A structural indicator Eurostat supports directly.

> Eurostat-only (URSSAF units are not on the same basis), so France is absent here.

In [ ]:
us  = m1.unit_size(p29[p29['source'] == 'eurostat'], pu)
fig = m1.fig_unit_size(us, save=str(fd.CONFIG.out('fig01_unit_size.png'))); plt.show()

## 8. France on its own terms (Rule 1)

URSSAF C29 by région, and the C27 two-level read that bounds the electrification proxy.

In [ ]:
fig = m1.fig_france_urssaf(save=str(fd.CONFIG.out('fig01_france.png'))); plt.show()

## 9. Maps

Three commitments, all visible in the legends:

1. **France is always drawn from URSSAF at NUTS-3 (département) level** — the finest level URSSAF supports, and finer than the NUTS-2 used elsewhere. It gets its own colour ramp so the two sources are never read as one measurement.
2. **Country borders are drawn thicker than region borders**, so a national boundary is distinguishable from an internal NUTS boundary at a glance.
3. **Every map carries a legend** saying what is shaded, from which source, at which level, and what the blanks mean.

> **On the France join.** French NUTS-3 codes are sequential *within* their parent region — `FR101` is Paris, `FR102` is Seine-et-Marne — so the digits have no relation to the département number. Matching on the code would draw a blank France without raising an error. The join is made on the département **name** instead (`fd.french_departement_to_nuts3`), and anything unmatched is reported rather than dropped.

> Folium needs a **trusted** notebook to render inline. If nothing appears, open the saved `.html`.

In [ ]:
m = m1.map_employment_europe(year=2023,
                            save=str(fd.CONFIG.out('map01_employment_europe.html')))
print('saved ->', os.path.abspath('outputs/map01_employment_europe.html'))
m

### 9b. Battery manufacturing in France

**What this can and cannot say.** APE **2720Z** is *fabrication de piles et d'accumulateurs électriques*. It does **not** separate lithium-ion traction cells from lead-acid starter batteries, so the class alone cannot identify EV production.

What it *can* do is show where battery employment sits and — more tellingly — **where it has grown since 2019**, because that growth is the gigafactory build-out. Nationally the class runs **4,369 (2019) → 7,463 (2024), +71%**.

Known EV cell plants (Billy-Berclau/Douvrin — ACC; Bourbourg/Dunkerque — Verkor; Douai — Envision AESC; Grenoble — Verkor) are ringed in black. Those are identified from **public knowledge, not from the data**, and the popups say so.

> No European equivalent is possible: Eurostat regional SBS publishes C27 only as an aggregate, with no 4-digit breakdown.

In [ ]:
m = m1.map_battery_france(year=2024,
                         save=str(fd.CONFIG.out('map01_battery_france.html')))
print('saved ->', os.path.abspath('outputs/map01_battery_france.html'))
m